# ARENA 0.2 — CNNs & ResNets Explainer

A line-by-line walkthrough of every code cell in the
exercises notebook, written for someone who has never
seen code before.

Each section mirrors one exercise cell. You get:
- A plain-English explanation of **what** and **why**
- The code with short, wrapped comments

---
## Quick Reference — Python Basics

| Syntax | Meaning |
|--------|---------|
| `import X` | Load library X |
| `import X as Y` | Load X, nickname it Y |
| `from X import Z` | Load just Z from X |
| `x = 5` | Store 5 in a box labelled x |
| `f"{x}"` | Insert x's value into a string |
| `def fn():` | Define a reusable function |
| `class Foo:` | Define a blueprint for objects |
| `self` | "this specific object" |
| `#` | Comment — ignored by Python |

---
# Part 1 — Setup

## Cell 1: Environment Detection

This cell figures out **where** the code is running
(Google Colab vs. your own machine) and downloads
the exercise files if needed.

In [ ]:
import os
import sys
from pathlib import Path

# Are we on Google Colab?
IN_COLAB = "google.colab" in sys.modules

# Folder names we'll need later
chapter = "chapter0_fundamentals"
repo = "ARENA_3.0"
branch = "main"

### Finding the root folder

The notebook builds a variable called `root` that
points to the top-level ARENA folder. It checks:

1. Colab → use `/content`
2. Not inside the repo → use `/root`
3. Inside the repo → walk up until you find it

**`os.getcwd()`** = "what folder am I in right now?"

**`Path.cwd().parents`** = list of parent folders
above the current one.

### The path setup lines

```python
sys.path.append(f"{root}/{chapter}/exercises")
os.chdir(f"{root}/{chapter}/exercises")
```

- `sys.path` is Python's search list for imports.
  We add the exercises folder so `import part2_cnns`
  works.
- `os.chdir(...)` changes the working directory,
  like `cd` in a terminal.

## Cell 2: Importing Libraries

Every `import` line loads a specific tool.
Here are the important ones:

In [ ]:
# --- Standard Python ---
import json           # read/write JSON files
from pathlib import Path

# --- Deep learning ---
import numpy as np    # fast array math ("np")
import torch as t     # PyTorch — the ML framework
import torch.nn as nn # neural-network building blocks
import torch.nn.functional as F  # one-shot functions

# --- Data & display ---
from torch import Tensor
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models, transforms
from tqdm.notebook import tqdm   # progress bars
from PIL import Image            # image files

# --- Local project files ---
import part2_cnns.tests as tests
import part2_cnns.utils as utils

### What is a Tensor?

A tensor is a multi-dimensional array of numbers:

| Dimensions | Name | Example |
|-----------|------|---------|
| 0 | scalar | `5` |
| 1 | vector | `[1, 2, 3]` |
| 2 | matrix | `[[1,2],[3,4]]` |
| 3 | 3D tensor | a cube of numbers |
| 4 | 4D tensor | a batch of images |

---
# Part 2 — Building Modules

## What is `nn.Module`?

Every layer in a neural network inherits from
`nn.Module`. Each module has:

- `__init__`: set it up (create weights, etc.)
- `forward`: what it **does** when data flows through

## ReLU — your first module

ReLU = "Rectified Linear Unit". It does one thing:

- Positive numbers → keep them
- Negative numbers → replace with zero

Example: `[-3, 5, -1, 8] → [0, 5, 0, 8]`

**Why?** Without it, stacking linear layers just
collapses into one linear layer. ReLU adds
"nonlinearity" so the network can learn curves.

In [ ]:
class ReLU(nn.Module):
    # Inherits from nn.Module.
    # No __init__ needed — no settings to store.

    def forward(self, x: Tensor) -> Tensor:
        # Compare each element to 0,
        # keep whichever is bigger.
        return t.maximum(x, t.tensor(0.0))

## Linear — the workhorse layer

A Linear layer computes weighted sums.

**Analogy:** Three judges score a skater on 5
criteria. Each judge weights the criteria
differently and produces one score.
That's `Linear(5, 3)` — 5 inputs → 3 outputs.

### Key concepts in this cell

- **`nn.Parameter`**: a tensor PyTorch knows to
  update during training. Regular tensors are just
  data; Parameters are "knobs" the network turns.

- **`super().__init__()`**: tells the parent class
  (`nn.Module`) to do its own setup first.

- **Kaiming init**: `1/√n` scaling keeps numbers
  from exploding or vanishing through layers.

In [ ]:
class Linear(nn.Module):
    def __init__(self, in_features, out_features,
                 bias=True):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features

        # Scale factor for initialization
        sf = 1 / np.sqrt(in_features)

        # Random weights in [-sf, sf]
        weight = sf * (2 * t.rand(
            out_features, in_features
        ) - 1)
        self.weight = nn.Parameter(weight)

        if bias:
            b = sf * (2 * t.rand(out_features) - 1)
            self.bias = nn.Parameter(b)
        else:
            self.bias = None

    def forward(self, x: Tensor) -> Tensor:
        # Matrix multiply: each output neuron is
        # a weighted sum of ALL inputs.
        x = einops.einsum(
            x, self.weight,
            "... in_f, out_f in_f -> ... out_f"
        )
        if self.bias is not None:
            x += self.bias
        return x

### What does `einops.einsum` do here?

The string describes the operation:

```
"... in_f, out_f in_f -> ... out_f"
```

- `... in_f` = input has batch dims + in_features
- `out_f in_f` = weight matrix shape
- `-> ... out_f` = output keeps batch dims,
  replaces in_features with out_features

For one image (784 pixels, 100 output neurons):
```
out[0] = x[0]*w[0,0] + x[1]*w[0,1] + ... + x[783]*w[0,783]
out[1] = x[0]*w[1,0] + x[1]*w[1,1] + ... + x[783]*w[1,783]
... (100 such sums)
```

## Flatten — reshape adapter

Convolutional layers output 3D data
(channels × height × width). Linear layers expect
a flat 1D list. Flatten bridges the gap.

Example: `(64, 32, 8, 8)` → `(64, 2048)`

The 64 is batch size (kept). Everything else
(32 × 8 × 8 = 2048) gets squashed flat.

In [ ]:
class Flatten(nn.Module):
    def __init__(self, start_dim=1, end_dim=-1):
        super().__init__()
        self.start_dim = start_dim
        self.end_dim = end_dim

    def forward(self, input: Tensor) -> Tensor:
        shape = input.shape

        # Handle negative index (-1 = last dim)
        start = self.start_dim
        end = self.end_dim
        if end < 0:
            end = len(shape) + end

        # Dims before, during, after the flatten
        left = shape[:start]
        right = shape[end + 1:]
        mid = t.prod(
            t.tensor(shape[start:end + 1])
        ).item()

        new_shape = left + (mid,) + right
        return t.reshape(input, new_shape)

## SimpleMLP — your first neural network!

```
Input (28×28 image)
  → Flatten    (784 numbers)
  → Linear     (784 → 100)
  → ReLU       (zero negatives)
  → Linear     (100 → 10 digit scores)
```

- **28×28 = 784**: MNIST images are 28 px square
- **100**: arbitrary hidden layer size
- **10**: one score per digit (0-9)

In [ ]:
class SimpleMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = Flatten()
        self.linear1 = Linear(28*28, 100)
        self.relu = ReLU()
        self.linear2 = Linear(100, 10)

    def forward(self, x: Tensor) -> Tensor:
        # Data flows through each layer in order.
        # Read inside-out: flatten first, linear2 last.
        return self.linear2(
            self.relu(
                self.linear1(
                    self.flatten(x)
                )
            )
        )
        # Equivalent to:
        #   x = self.flatten(x)
        #   x = self.linear1(x)
        #   x = self.relu(x)
        #   x = self.linear2(x)
        #   return x

---
# Part 3 — Training

## Transforms & Data Loading

Before a network can look at images, they must be
converted to tensors and preprocessed.

In [ ]:
MNIST_TRANSFORM = transforms.Compose([
    transforms.ToTensor(),
    # Converts image to tensor, scales 0-255 → 0-1

    transforms.Normalize(0.1307, 0.3081),
    # Centers data: pixel = (pixel - mean) / std
    # 0.1307 = mean of all MNIST pixels
    # 0.3081 = std deviation
    # Networks learn better with centered data.
])

In [ ]:
def get_mnist(trainset_size=10_000,
              testset_size=1_000):
    """Download MNIST and take a subset."""

    mnist_train = datasets.MNIST(
        exercises_dir / "data",
        train=True,    # 60k training images
        download=True,
        transform=MNIST_TRANSFORM,
    )
    mnist_test = datasets.MNIST(
        exercises_dir / "data",
        train=False,   # 10k test images
        download=True,
        transform=MNIST_TRANSFORM,
    )

    # Take smaller subsets for speed
    return (
        Subset(mnist_train, range(trainset_size)),
        Subset(mnist_test, range(testset_size)),
    )

### DataLoader

A DataLoader serves data in **batches**:

- `batch_size=64`: feed 64 images at a time
- `shuffle=True`: randomize order each epoch
  (prevents memorizing the sequence)

```python
trainloader = DataLoader(
    trainset, batch_size=64, shuffle=True
)
```

### Device: CPU vs GPU

GPUs do thousands of operations in parallel,
making training 10-100× faster than CPU.

```python
device = t.device(
    "mps" if t.backends.mps.is_available()
    else "cuda" if t.cuda.is_available()
    else "cpu"
)
```

- `mps` = Apple Silicon GPU
- `cuda` = NVIDIA GPU
- `cpu` = fallback

## The Training Loop

This is where learning happens. The recipe:

1. Show the model a batch of images
2. Model makes predictions (probably wrong at first)
3. Measure how wrong (**loss**)
4. Compute which way to adjust weights
   (**backpropagation**)
5. Nudge the weights (**optimizer step**)
6. Repeat thousands of times

In [ ]:
model = SimpleMLP().to(device)
# .to(device) moves model to GPU

optimizer = t.optim.Adam(
    model.parameters(), lr=1e-3
)
# Adam = smart weight adjuster
# lr = learning rate = step size (0.001)
# model.parameters() = all learnable weights

loss_list = []

for epoch in range(3):
    for imgs, labels in tqdm(mnist_trainloader):

        # Move data to same device as model
        imgs = imgs.to(device)
        labels = labels.to(device)

        # 1. Forward pass — make predictions
        logits = model(imgs)
        # logits shape: (batch, 10)
        # Raw scores, one per digit class

        # 2. Compute loss — how wrong?
        loss = F.cross_entropy(logits, labels)
        # Internally: softmax → probability of
        # correct class → negative log.
        # Low loss = confident and correct.

        # 3. Backward pass — compute gradients
        loss.backward()
        # For each weight: "how much would loss
        # change if I nudged this weight?"

        # 4. Update weights
        optimizer.step()

        # 5. Zero gradients for next batch
        optimizer.zero_grad()
        # Without this, gradients accumulate!

        loss_list.append(loss.item())

### What each step does

| Step | Code | Purpose |
|------|------|---------|
| Forward | `model(imgs)` | Predict |
| Loss | `F.cross_entropy(...)` | Measure error |
| Backward | `loss.backward()` | Find gradients |
| Step | `optimizer.step()` | Update weights |
| Zero | `optimizer.zero_grad()` | Reset for next batch |

## `@dataclass` — organizing settings

A dataclass bundles related settings together
instead of passing 10 separate arguments.

In [ ]:
@dataclass
class SimpleMLPTrainingArgs:
    batch_size: int = 64
    epochs: int = 3
    learning_rate: float = 1e-3

    # @dataclass auto-generates __init__ from
    # the fields above. You can override defaults:
    #   args = SimpleMLPTrainingArgs(epochs=10)

## Validation Loop

Training teaches the model. Validation **tests** it
on images it has never seen.

| | Training | Validation |
|---|----------|------------|
| Purpose | Learn | Test |
| Data | Train set | Test set |
| Metric | Loss | Accuracy |
| Gradients | Yes | No |
| Updates weights? | Yes | No |

In [ ]:
# Inside the training function, after each epoch:

num_correct = 0

for imgs, labels in mnist_testloader:
    imgs = imgs.to(device)
    labels = labels.to(device)

    with t.inference_mode():
        # "Don't track gradients — just observing"
        logits = model(imgs)

    predictions = t.argmax(logits, dim=1)
    # argmax = index of highest score
    # e.g. scores [.1,.1,.8,...] → prediction = 2

    num_correct += (
        (predictions == labels).sum().item()
    )
    # True where correct, .sum() counts them

accuracy = num_correct / len(mnist_testset)
# e.g. 950/1000 = 95%

---
# Part 4 — Convolutions

## What is a convolution?

A small grid of weights (the **kernel**, e.g. 3×3)
slides across the image. At each position it
computes a weighted sum of the pixels underneath.

The result is a **feature map** — a new image that
highlights certain patterns (edges, textures, etc.).

### Key parameters

| Parameter | Meaning |
|-----------|---------|
| `in_channels` | Input types (1=gray, 3=RGB) |
| `out_channels` | Number of filters |
| `kernel_size` | Window size (3 = 3×3) |
| `stride` | How far to slide (default 1) |
| `padding` | Zeros added around border |

In [ ]:
class Conv2d(nn.Module):
    def __init__(self, in_channels, out_channels,
                 kernel_size, stride=1, padding=0):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.kernel_size = kernel_size
        self.stride = stride
        self.padding = padding

        # Kaiming initialization
        fan_in = in_channels * kernel_size ** 2
        sf = 1 / fan_in ** 0.5
        weight = sf * (2 * t.rand(
            out_channels, in_channels,
            kernel_size, kernel_size,
        ) - 1)
        self.weight = nn.Parameter(weight)

    def forward(self, x: Tensor) -> Tensor:
        return F.conv2d(
            x, self.weight,
            stride=self.stride,
            padding=self.padding,
        )

## MaxPool2d — shrinking the image

Tiles the image with non-overlapping windows
(e.g. 2×2) and keeps only the **largest** value
in each tile. Halves both dimensions.

- Reduces computation
- Makes the network robust to small shifts

In [ ]:
class MaxPool2d(nn.Module):
    def __init__(self, kernel_size,
                 stride=None, padding=1):
        super().__init__()
        self.kernel_size = kernel_size
        self.stride = stride
        self.padding = padding

    def forward(self, x: Tensor) -> Tensor:
        return F.max_pool2d(
            x,
            kernel_size=self.kernel_size,
            stride=self.stride,
            padding=self.padding,
        )

## BatchNorm2d — stabilizing training

After each conv layer, normalizes the data so each
channel has mean ≈ 0 and variance ≈ 1. Then lets
the network learn a scale and shift.

**Parameters** (learned via backprop):
- `weight` — scale factor per channel
- `bias` — shift per channel

**Buffers** (updated by formula, not gradients):
- `running_mean` — long-term average
- `running_var` — long-term variance

In **training mode**: uses current batch stats.
In **eval mode**: uses stored running stats.

In [ ]:
class BatchNorm2d(nn.Module):
    def __init__(self, num_features,
                 eps=1e-05, momentum=0.1):
        super().__init__()
        self.num_features = num_features
        self.eps = eps
        self.momentum = momentum

        # Learnable scale and shift
        self.weight = nn.Parameter(
            t.ones(num_features)
        )
        self.bias = nn.Parameter(
            t.zeros(num_features)
        )

        # Running statistics (not trained)
        self.register_buffer(
            "running_mean", t.zeros(num_features)
        )
        self.register_buffer(
            "running_var", t.ones(num_features)
        )
        self.register_buffer(
            "num_batches_tracked", t.tensor(0)
        )

---
# Part 5 — ResNets

## Sequential — chaining modules

A pipeline: data goes in one end and passes
through each module in order.

In [ ]:
class Sequential(nn.Module):
    def __init__(self, *modules):
        super().__init__()
        for i, mod in enumerate(modules):
            self._modules[str(i)] = mod

    def forward(self, x: Tensor) -> Tensor:
        for mod in self._modules.values():
            x = mod(x)
        return x

## ResidualBlock — the key innovation

Instead of just transforming the input, a residual
block **adds the input back** to the output:

```
output = F(input) + input
```

This "skip connection" lets gradients flow directly
backward through very deep networks (50+ layers)
without vanishing.

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, in_feats, out_feats,
                 first_stride=1):
        super().__init__()

        same_shape = (
            first_stride == 1
            and in_feats == out_feats
        )

        # Left branch: the transformation
        self.left = Sequential(
            Conv2d(in_feats, out_feats,
                   kernel_size=3,
                   stride=first_stride,
                   padding=1),
            BatchNorm2d(out_feats),
            ReLU(),
            Conv2d(out_feats, out_feats,
                   kernel_size=3, stride=1,
                   padding=1),
            BatchNorm2d(out_feats),
        )

        # Right branch: skip connection
        if same_shape:
            self.right = nn.Identity()
        else:
            # 1x1 conv to match shapes
            self.right = Sequential(
                Conv2d(in_feats, out_feats,
                       kernel_size=1,
                       stride=first_stride),
                BatchNorm2d(out_feats),
            )

    def forward(self, x: Tensor) -> Tensor:
        # THE RESIDUAL CONNECTION:
        return self.left(x) + self.right(x)

## ResNet34 — the full architecture

```
Input (3, 224, 224)
  Conv7×7 + BN + ReLU + MaxPool → (64, 56, 56)
  3 ResidualBlocks (64 → 64)    → (64, 56, 56)
  4 ResidualBlocks (64 → 128)   → (128, 28, 28)
  6 ResidualBlocks (128 → 256)  → (256, 14, 14)
  3 ResidualBlocks (256 → 512)  → (512, 7, 7)
  AveragePool                    → (512)
  Linear                         → (1000 classes)
```

In [ ]:
class BlockGroup(nn.Module):
    def __init__(self, n_blocks, in_feats,
                 out_feats, first_stride=1):
        super().__init__()
        blocks = [ResidualBlock(
            in_feats, out_feats, first_stride
        )]
        for _ in range(1, n_blocks):
            blocks.append(
                ResidualBlock(out_feats, out_feats)
            )
        self.blocks = Sequential(*blocks)

    def forward(self, x: Tensor) -> Tensor:
        return self.blocks(x)

In [ ]:
class ResNet34(nn.Module):
    def __init__(self,
        n_blocks_per_group=[3, 4, 6, 3],
        out_features_per_group=[64, 128, 256, 512],
        first_strides_per_group=[1, 2, 2, 2],
        n_classes=1000,
    ):
        super().__init__()

        # Initial layers
        self.in_layers = Sequential(
            Conv2d(3, 64, kernel_size=7,
                   stride=2, padding=3),
            BatchNorm2d(64),
            ReLU(),
            MaxPool2d(kernel_size=3,
                      stride=2, padding=1),
        )

        # Build residual block groups
        in_f = [64] + out_features_per_group[:-1]
        self.residual_layers = Sequential(*[
            BlockGroup(n, i, o, s)
            for n, i, o, s in zip(
                n_blocks_per_group,
                in_f,
                out_features_per_group,
                first_strides_per_group,
            )
        ])

        # Final layers
        self.out_layers = Sequential(
            AveragePool(),
            Flatten(),
            Linear(512, n_classes),
        )

    def forward(self, x: Tensor) -> Tensor:
        x = self.in_layers(x)
        x = self.residual_layers(x)
        x = self.out_layers(x)
        return x

## Using Pretrained Weights

Instead of training ResNet34 for days, we copy
weights from a model PyTorch already trained.

In [ ]:
def copy_weights(my_resnet, pretrained_resnet):
    """Copy pretrained weights into our model."""
    mydict = my_resnet.state_dict()
    predict = pretrained_resnet.state_dict()

    # Map our param names to their values
    new_dict = {
        my_key: pre_val
        for (my_key, _), (_, pre_val)
        in zip(mydict.items(), predict.items())
    }
    my_resnet.load_state_dict(new_dict)
    return my_resnet

## Making Predictions

In [ ]:
@t.inference_mode()
def predict(model, images):
    model.eval()  # eval mode (affects BatchNorm)

    logits = model(images)
    # shape: (batch, 1000)

    probs = logits.softmax(dim=-1)
    # Convert scores → probabilities (sum to 1)

    top_probs, top_indices = probs.max(dim=-1)
    # Highest probability and its class index

    return top_probs, top_indices

---
# Part 6 — Bonus: Feature Extraction

Take a pretrained model and repurpose it for a
different task by:

1. **Freeze** all layers (lock the weights)
2. **Replace** the last layer for your task
3. **Train** only the new last layer

Early layers learn universal features (edges,
textures). Only the final layer is task-specific.

In [ ]:
def get_resnet_for_feature_extraction(n_classes):
    my_resnet = ResNet34()
    pretrained = models.resnet34(
        weights=models.ResNet34_Weights.IMAGENET1K_V1
    )
    my_resnet = copy_weights(my_resnet, pretrained)

    # Freeze ALL weights
    my_resnet.requires_grad_(False)

    # Replace final layer (unfrozen by default)
    my_resnet.out_layers[-1] = Linear(512, n_classes)

    return my_resnet

---
# Part 7 — Bonus: Convolutions From Scratch

## How tensors live in memory

A 2D tensor is stored as a **flat 1D block**.
PyTorch uses **strides** to navigate it:

```
[[0, 1, 2],     stored as [0,1,2,3,4,5]
 [3, 4, 5]]     shape=(2,3)  stride=(3,1)
```

- stride = (3, 1) means:
  - Skip 3 elements to go down a row
  - Skip 1 element to go right a column

`as_strided` lets you create **views** with custom
strides — no data is copied, just a different way
of reading the same memory.

In [ ]:
test_input = t.tensor([
    [0, 1, 2, 3, 4],
    [5, 6, 7, 8, 9],
    [10, 11, 12, 13, 14],
    [15, 16, 17, 18, 19],
], dtype=t.float)

# shape = (4, 5),  stride = (5, 1)
# Memory: [0,1,2,3,4, 5,6,7,8,9, 10,..., 19]

### Stride exercises

Fill in `size` and `stride` to get each output.

In [ ]:
# First row: [0, 1, 2, 3, 4]
# size=(5,)  stride=(1,)

# First column: [0, 5, 10, 15]
# size=(4,)  stride=(5,)

# Top-left 2x3: [[0,1,2],[5,6,7]]
# size=(2,3)  stride=(5,1)

# Diagonal: [0, 6, 12, 18]
# size=(4,)  stride=(6,)
#   6 = row_stride + col_stride = 5 + 1

# Repeated elements: [[0,0,0],[11,11,11]]
# size=(2,3)  stride=(11,0)
#   stride=0 means "don't move" → repeats!

## Conv1d from scratch

The trick: use `as_strided` to create overlapping
windows, then multiply by the kernel and sum.

In [ ]:
def conv1d_minimal_simple(x, weights):
    """1D conv: single input, single kernel."""
    w = x.shape[0]
    kw = weights.shape[0]
    out_w = w - kw + 1

    # Create overlapping windows
    x_strided = x.as_strided(
        size=(out_w, kw),
        stride=(1, 1),
    )
    # Row 0: x[0], x[1], x[2]
    # Row 1: x[1], x[2], x[3]
    # Row 2: x[2], x[3], x[4]

    return (x_strided * weights).sum(dim=1)

In [ ]:
def conv2d_minimal(x, weights):
    """2D conv using as_strided."""
    batch, in_ch, h, w = x.shape
    out_ch, _, kh, kw = weights.shape
    oh = h - kh + 1
    ow = w - kw + 1

    # 2D windowed view
    x_strided = x.as_strided(
        size=(batch, in_ch, oh, ow, kh, kw),
        stride=(
            x.stride(0), x.stride(1),
            x.stride(2), x.stride(3),
            x.stride(2), x.stride(3),
        ),
    )

    return einops.einsum(
        x_strided, weights,
        "b ic oh ow kh kw,"
        " oc ic kh kw"
        " -> b oc oh ow",
    )

## Padding and stride

- **Padding** adds zeros around the border so the
  output stays the same size as the input.
- **Stride > 1** makes the kernel skip positions,
  shrinking the output.

In [ ]:
def pad2d(x, left, right, top, bottom, val=0.0):
    """Pad a 4D tensor with val."""
    b, c, h, w = x.shape
    out = t.full(
        (b, c, top+h+bottom, left+w+right),
        val,
    )
    out[..., top:top+h, left:left+w] = x
    return out

In [ ]:
def maxpool2d(x, kernel_size, stride=None,
              padding=0):
    """Max pooling using as_strided."""
    kh, kw = force_pair(kernel_size)
    if stride is None:
        stride = (kh, kw)
    sh, sw = force_pair(stride)
    ph, pw = force_pair(padding)

    if ph > 0 or pw > 0:
        x = pad2d(x, pw, pw, ph, ph, float("-inf"))
        # Pad with -inf so padded values are
        # never the maximum

    b, c, h, w = x.shape
    oh = (h - kh) // sh + 1
    ow = (w - kw) // sw + 1

    x_strided = x.as_strided(
        size=(b, c, oh, ow, kh, kw),
        stride=(
            x.stride(0), x.stride(1),
            x.stride(2)*sh, x.stride(3)*sw,
            x.stride(2), x.stride(3),
        ),
    )
    return x_strided.amax(dim=(-2, -1))

---
# Summary

## Building blocks

| Module | What it does | Learnable? |
|--------|-------------|------------|
| ReLU | Zero out negatives | No |
| Linear | Weighted sums | Yes |
| Flatten | Reshape to flat | No |
| Conv2d | Sliding window features | Yes |
| MaxPool2d | Shrink by keeping max | No |
| BatchNorm2d | Normalize per channel | Yes |
| AveragePool | Average across space | No |

## Training loop steps

1. **Forward**: data → model → predictions
2. **Loss**: measure how wrong
3. **Backward**: compute gradients
4. **Step**: update weights
5. **Zero**: reset gradients
6. **Repeat**

## Key PyTorch concepts

| Concept | Meaning |
|---------|---------|
| `nn.Parameter` | Tensor that gets trained |
| `forward()` | What a module does to data |
| `.to(device)` | Move to GPU or CPU |
| `loss.backward()` | Compute all gradients |
| `optimizer.step()` | Update all weights |
| `inference_mode()` | Disable gradient tracking |
| `as_strided` | Custom view of memory |